In [4]:
# Cell 1: System Imports & Audio/Brightness Setup
import os
import time
import math
import cv2
import numpy as np
import pyautogui
import screen_brightness_control as sbc

# Disable PyAutoGUI fail-safe for smooth cursor movement
pyautogui.FAILSAFE = False
screen_w, screen_h = pyautogui.size()

# System Audio Control Setup (Pycaw)
try:
    from ctypes import POINTER, cast
    from comtypes import CLSCTX_ALL
    from pycaw.pycaw import AudioUtilities, IAudioEndpointVolume

    devices = AudioUtilities.GetSpeakers()
    interface = devices.Activate(IAudioEndpointVolume._iid_, CLSCTX_ALL, None)
    volume = cast(interface, POINTER(IAudioEndpointVolume))
    vol_range = volume.GetVolumeRange()
    min_vol, max_vol = vol_range[0], vol_range[1]
    has_audio = True
    print("Audio System Initialized Successfully!")
except Exception as e:
    has_audio = False
    print("Audio control warning:", e)

Audio control warning: 'AudioDevice' object has no attribute 'Activate'


In [5]:
# Cell 2: Initialize MediaPipe Hand Tracking
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_draw = mp.solutions.drawing_utils

# Setup Detector for hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)

print("MediaPipe Hands Initialized Successfully!")

MediaPipe Hands Initialized Successfully!


In [8]:
# Cell 3: Final Smooth Gesture Control (Media Keys for Volume)
cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

print("Starting Final Gesture Control... Press 'q' to exit.")

frame_count = 0
last_volume_time = 0

while cap.isOpened():
    success, img = cap.read()
    if not success:
        print("Ignoring empty camera frame.")
        continue

    img = cv2.flip(img, 1)
    img_h, img_w, _ = img.shape
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    results = hands.process(img_rgb)
    frame_count += 1

    if results.multi_hand_landmarks and results.multi_handedness:
        for hand_idx, hand_landmarks in enumerate(results.multi_hand_landmarks):
            classification = results.multi_handedness[hand_idx].classification[0].label
            mp_draw.draw_landmarks(img, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            
            lm_list = []
            for id, lm in enumerate(hand_landmarks.landmark):
                cx, cy = int(lm.x * img_w), int(lm.y * img_h)
                lm_list.append((id, cx, cy))
                
            if len(lm_list) != 0:
                # --- LEFT HAND: Volume & Brightness ---
                if classification == 'Left':
                    x_thumb, y_thumb = lm_list[4][1], lm_list[4][2]
                    x_index, y_index = lm_list[8][1], lm_list[8][2]
                    x_middle, y_middle = lm_list[12][1], lm_list[12][2]
                    
                    cv2.circle(img, (x_thumb, y_thumb), 8, (0, 255, 255), cv2.FILLED)
                    cv2.circle(img, (x_index, y_index), 8, (0, 255, 255), cv2.FILLED)
                    cv2.circle(img, (x_middle, y_middle), 8, (255, 0, 255), cv2.FILLED)
                    
                    vol_length = math.hypot(x_index - x_thumb, y_index - y_thumb)
                    bright_length = math.hypot(x_middle - x_thumb, y_middle - y_thumb)
                    
                    # Volume Control using Media Keys (Throttled with time delay)
                    current_time = time.time()
                    if current_time - last_volume_time > 0.15:
                        if vol_length > 120:
                            pyautogui.press('volumeup')
                            last_volume_time = current_time
                        elif vol_length < 50:
                            pyautogui.press('volumedown')
                            last_volume_time = current_time

                    # Brightness Control
                    if frame_count % 5 == 0:
                        try:
                            brightness = np.interp(bright_length, [30, 180], [0, 100])
                            sbc.set_brightness(int(brightness))
                        except:
                            pass
                            
                    cv2.putText(img, 'Left Hand: Volume & Brightness', (30, 50), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

                # --- RIGHT HAND: Cursor & Click ---
                elif classification == 'Right':
                    x_index, y_index = lm_list[8][1], lm_list[8][2]
                    x_thumb, y_thumb = lm_list[4][1], lm_list[4][2]
                    
                    screen_x = np.interp(x_index, [50, img_w - 50], [0, screen_w])
                    screen_y = np.interp(y_index, [50, img_h - 50], [0, screen_h])
                    pyautogui.moveTo(screen_x, screen_y)
                    
                    cv2.circle(img, (x_index, y_index), 8, (255, 0, 0), cv2.FILLED)
                    
                    click_dist = math.hypot(x_thumb - x_index, y_thumb - y_index)
                    if click_dist < 30:
                        cv2.circle(img, (x_index, y_index), 12, (0, 0, 255), cv2.FILLED)
                        pyautogui.click()
                        time.sleep(0.1)
                        
                    cv2.putText(img, 'Right Hand: Mouse Active', (30, 80), 
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    cv2.imshow('Hand Gesture Control', img)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Gesture Control Stopped Safely.")

Starting Final Gesture Control... Press 'q' to exit.
Gesture Control Stopped Safely.
